In [4]:
from read_raw_data import read_dat_file

# write a function called get reference signal that takes path to a file and gives you a start and stop index for the reference signal
# this function should take the center freq of rx as input and also the sample rate. it should also take tx frequency as input so it knows the bin to look for high power in. 
# when parsing the signal, when the power is high for first time, start index is set. when power is low for first time after start index is set, stop index is set.
# return start and stop index.

iq0_file_path = '/home/ishani/LeoPNT/USRP_data/1_27/time_15_15_freq_11_325G_bw_2.0M_1706391792.6762946_inrx0.dat'
iq_path = [iq0_file_path, iq0_file_path.replace('rx0', 'rx1'), iq0_file_path.replace('rx0', 'rx2')]

# iq = [read_dat_file(iq_path[0], None, None), read_dat_file(iq_path[1], None, None), read_dat_file(iq_path[2], None, None)]

# print(iq[0].shape, iq[1].shape, iq[2].shape)

In [5]:
from turtle import st
import numpy as np
from matplotlib import pyplot as plt
from utils.ref_sig import get_reference_signal_indexes
# Set a higher value for agg.path.chunksize
plt.rcParams['agg.path.chunksize'] = 200

file_path = iq0_file_path
center_freq_rx = 11.325e9  # Replace with the actual center frequency of your receiver
sample_rate = 2e6  # Replace with the actual sample rate of your signal
tx_freq = 11.3252e9  # Replace with the actual transmit frequency
nfft = 2**15  # Replace with the fft size you want to use
num_read_samples = nfft  # Replace with the number of samples you want to read at a time
start_from = None
plt.close('all')
plt.figure(figsize=(20, 15))
i = 1
start_index_arr = []
stop_index_arr = []
mean_phase_arr = []
while True:
  
    start_index, stop_index = get_reference_signal_indexes(file_path, center_freq_rx, sample_rate, tx_freq, num_read_samples, nfft, start_from)
    print("Start Index:", start_index)
    print("Stop Index:", stop_index)
    if start_index is None or stop_index is None:
        break   
    print("Ref siganl lasts for:", str((stop_index-start_index)/2e6)+" s")
    if start_from is not None:
        print("Capture lasts for:", str((stop_index-start_from)/2e6)+" s")
    start_from = int(stop_index+1e6) # setting for next iteration
    begin_index = int(start_index+2**15)
    end_index = int(stop_index-2**15)
    iq = [read_dat_file(iq_path[0], end_index-begin_index, begin_index), read_dat_file(iq_path[1], end_index-begin_index, begin_index), read_dat_file(iq_path[2], end_index-begin_index, begin_index)]
    iq_ = [iq[0], iq[1], iq[2]]
    # where iq is zero, replace with Nan
    iq_[0] = np.where(iq_[0] == 0, 1e-30, iq_[0])
    iq_[1] = np.where(iq_[1] == 0, 1e-30, iq_[1])
    iq_[2] = np.where(iq_[2] == 0, 1e-30, iq_[2])
    
    del_phi = [(np.angle(iq_[0]/iq_[1])), (np.angle(iq_[0]/iq_[2])), (np.angle(iq_[1]/iq_[2]))]
    start_index_arr.append(begin_index)
    stop_index_arr.append(end_index)
    mean_phase_arr.append([np.mean(del_phi[0]), np.mean(del_phi[1]), np.mean(del_phi[2])])
    print("Mean Phase:", [np.mean(del_phi[0]), np.mean(del_phi[1]), np.mean(del_phi[2])])
    # if i%3 == 0:
    #     plt.subplot(6, 4, int(i/3))
    #     plt.plot(del_phi[0])
    #     plt.plot(del_phi[1])
    #     plt.plot(del_phi[2])
    #     plt.title(str((begin_index-13762560)/2e6)+' s')
    #     plt.ylim([-1.25*np.pi, 1.25*np.pi])
    #     plt.grid()
    # i+=1

# plt.legend(['rx0-rx1', 'rx0-rx2', 'rx1-rx2'], loc="upper right")


# plt.savefig('del_phi.png')

tx_bin:  1638
Start Index: 13762560
Stop Index: 24739840
Ref siganl lasts for: 5.48864 s
Mean Phase: [-0.40445328, 1.12014, 1.524591]
tx_bin:  1638
Start Index: 33702464
Stop Index: 41894464
Ref siganl lasts for: 4.096 s
Capture lasts for: 8.077312 s
Mean Phase: [-0.4302309, 1.0928377, 1.5230684]
tx_bin:  1638
Start Index: 51709056
Stop Index: 62588032
Ref siganl lasts for: 5.439488 s
Capture lasts for: 9.846784 s
Mean Phase: [-0.45314032, 1.0685612, 1.5217028]
tx_bin:  1638
Start Index: 73778880
Stop Index: 80627392
Ref siganl lasts for: 3.424256 s
Capture lasts for: 8.51968 s
Mean Phase: [-0.470259, 1.0498662, 1.5201255]
tx_bin:  1638
Start Index: 127699200
Stop Index: 133368064
Ref siganl lasts for: 2.834432 s
Capture lasts for: 25.870336 s
Mean Phase: [-0.50703114, 1.011095, 1.5181249]
tx_bin:  1638
Start Index: 187779904
Stop Index: 192334656
Ref siganl lasts for: 2.277376 s
Capture lasts for: 28.983296 s
Mean Phase: [-0.5353933, 0.98110217, 1.516495]
tx_bin:  1638
Start Index: 21

<Figure size 2000x1500 with 0 Axes>

In [9]:
phase_drift_arr = []
for i in range(len(mean_phase_arr)-1):
    phase_drift_arr.append([(mean_phase_arr[i+1][0]-mean_phase_arr[i][0])/((start_index_arr[i+1]-stop_index_arr[i])/2e6), (mean_phase_arr[i+1][1]-mean_phase_arr[i][1])/((start_index_arr[i+1]-stop_index_arr[i])/2e6), (mean_phase_arr[i+1][2]-mean_phase_arr[i][2])/((start_index_arr[i+1]-stop_index_arr[i])/2e6)])

phase_drift_arr = np.array(phase_drift_arr)
print("Phase Drift:", phase_drift_arr)
print("Mean Phase Drift:", np.mean(phase_drift_arr,axis=0))
print("Std Phase Drift:", np.std(phase_drift_arr,axis=0))

Phase Drift: [[-1.83690374e-03 -6.04824575e-03 -3.37287121e-04]
 [-1.58982701e-03 -4.91420657e-03 -2.76446139e-04]
 [-1.18662261e-03 -3.32167006e-03 -2.80242414e-04]
 [-1.23554699e-03 -1.64502915e-03 -8.48826059e-05]
 [-8.78494350e-04 -1.10111318e-03 -5.98394686e-05]
 [-7.72226001e-04 -1.22142961e-03 -1.39039162e-04]
 [-3.90631593e-04 -6.33176669e-04 -4.38694957e-05]
 [-4.19696353e-04 -8.37604142e-04 -8.52797505e-05]
 [-3.70282383e-04 -6.80707162e-04 -6.42421505e-05]
 [-1.08000518e-04 -1.00683990e-04  5.96693955e-05]]
Mean Phase Drift: [-0.00087882 -0.00205039 -0.00013115]
Std Phase Drift: [0.00054394 0.00191598 0.00011989]
